# XGBoost hill-climbing ensemble analysis

> This notebook is analysis-only. Use the CLI runtime in hill_climbing_ensemble to run or resume search.

Review Optuna trial history, accepted-member progression, and final ensemble artifacts while the background run is in progress.

In [ ]:
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns

from helper_functions.gradient_boosting_search import summarize_with_ci

## 1. Monitoring configuration

In [ ]:
OPTUNA_STORAGE_FILE = Path('../data/results/08-xgboost-hillclimb-optuna.db')
STUDY_NAME = 'hillclimb_08'

ENSEMBLE_LOG_FILE = Path('../data/results/08-xgboost-hillclimb-log.pkl')
ENSEMBLE_MODEL_FILE = Path('../data/results/08-xgboost-ensemble.pkl')
RUN_STATE_FILE = Path('../data/results/08-xgboost-hillclimb-run-state.pkl')
FINAL_SUBMISSION_FILE = Path('../data/submission.csv')

MAX_RECENT_ACCEPTS_TO_SHOW = 12
PLOT_ROLLING_WINDOW = 10

## 2. Load run artifacts

In [ ]:
if not OPTUNA_STORAGE_FILE.exists():
    raise FileNotFoundError(f'Optuna storage file not found: {OPTUNA_STORAGE_FILE}')

study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=f'sqlite:///{OPTUNA_STORAGE_FILE}',
    direction='maximize',
    load_if_exists=True,
)

with ENSEMBLE_MODEL_FILE.open('rb') as handle:
    ensemble_payload = pickle.load(handle)

if ENSEMBLE_LOG_FILE.exists():
    with ENSEMBLE_LOG_FILE.open('rb') as handle:
        hill_log = pickle.load(handle)
else:
    hill_log = []

if RUN_STATE_FILE.exists():
    with RUN_STATE_FILE.open('rb') as handle:
        run_state = pickle.load(handle)
else:
    run_state = {}

if FINAL_SUBMISSION_FILE.exists():
    submission_df = pd.read_csv(FINAL_SUBMISSION_FILE)
else:
    submission_df = pd.DataFrame()

accepted_specs = ensemble_payload.get('accepted_specs', [])
trials_df = study.trials_dataframe(attrs=('number', 'value', 'state', 'datetime_start', 'datetime_complete', 'duration'))

print(f'Study: {STUDY_NAME}')
print(f'Trials recorded: {len(trials_df)}')
print(f'Accepted models: {len(accepted_specs)}')
print(f'Hill log records: {len(hill_log)}')
print(f'Run state checkpoint exists: {bool(run_state)}')
print(f'Final submission rows: {len(submission_df)}')

## 3. Build analysis views

In [ ]:
log_df = pd.DataFrame(hill_log)

if not log_df.empty:
    log_df = log_df.sort_values('proposal_index').reset_index(drop=True)
    log_df['accepted_count_running'] = log_df['accepted'].astype(int).cumsum()
    log_df['best_median_after'] = np.where(
        log_df['accepted'],
        log_df['proposed_median'],
        np.nan,
    )
    log_df['best_median_after'] = log_df['best_median_after'].ffill().fillna(0.0)
    log_df['rolling_proposed_median'] = log_df['proposed_median'].rolling(
        PLOT_ROLLING_WINDOW,
        min_periods=1,
    ).mean()
    accepted_df = log_df[log_df['accepted']].copy()
else:
    accepted_df = pd.DataFrame()

if accepted_specs:
    accepted_specs_df = pd.DataFrame(
        [
            {
                'accepted_index': idx,
                'weight': spec.get('weight', np.nan),
                'row_fraction': spec.get('row_fraction', np.nan),
                'feature_fraction': spec.get('feature_fraction', np.nan),
                'feature_count': len(spec.get('feature_columns', [])),
            }
            for idx, spec in enumerate(accepted_specs, start=1)
        ]
    )
    accepted_params_df = pd.DataFrame([spec.get('params', {}) for spec in accepted_specs])
else:
    accepted_specs_df = pd.DataFrame()
    accepted_params_df = pd.DataFrame()

trials_completed = trials_df[trials_df['state'] == 'COMPLETE'].copy() if not trials_df.empty else pd.DataFrame()

## 4. Current run summary

In [ ]:
if log_df.empty:
    print('No hill-climb log entries available yet.')
else:
    latest = log_df.iloc[-1]
    print(f"Latest proposal index: {int(latest['proposal_index'])}")
    print(f"Accepted models so far: {int(latest['accepted_count_running'])}")
    print(f"Current best median BA: {latest['best_median_after']:.5f}")
    print(f"Latest proposal median BA: {latest['proposed_median']:.5f}")

if not accepted_df.empty:
    print('\nRecent accepted proposals')
    recent_cols = [
        'proposal_index',
        'accepted_count_running',
        'proposed_median',
        'delta',
        'feature_count',
    ]
    print(accepted_df[recent_cols].tail(MAX_RECENT_ACCEPTS_TO_SHOW).to_string(index=False))

if not accepted_specs_df.empty:
    print('\nAccepted member sampling summary')
    print(accepted_specs_df.describe().to_string())

final_summary = ensemble_payload.get('final_cv_summary')
if final_summary:
    print('\nFinal CV summary from payload')
    print(f"Mean balanced accuracy:   {final_summary['mean']:.5f}")
    print(f"Median balanced accuracy: {final_summary['median']:.5f}")
    print(f"Std balanced accuracy:    {final_summary['std']:.5f}")
    fold_scores = final_summary.get('fold_scores', [])
    if fold_scores:
        ci_summary = summarize_with_ci(fold_scores)
        print(
            f"Mean 95% CI:   ({ci_summary['mean']['ci_lower']:.5f}, {ci_summary['mean']['ci_upper']:.5f})"
        )
        print(
            f"Median 95% CI: ({ci_summary['median']['ci_lower']:.5f}, {ci_summary['median']['ci_upper']:.5f})"
        )

if not submission_df.empty:
    print('\nCurrent submission label counts')
    print(submission_df['health_condition'].value_counts().to_string())

## 5. Trial and acceptance plots

In [ ]:
if log_df.empty:
    print('No run history to plot yet.')
else:
    plt.figure(figsize=(12, 5))
    plt.plot(log_df['proposal_index'], log_df['proposed_median'], alpha=0.35, label='proposed median BA')
    plt.plot(log_df['proposal_index'], log_df['best_median_after'], linewidth=2.0, label='best median BA (accepted)')
    plt.plot(
        log_df['proposal_index'],
        log_df['rolling_proposed_median'],
        linestyle='--',
        linewidth=1.5,
        label=f'rolling mean ({PLOT_ROLLING_WINDOW})',
    )
    plt.xlabel('Proposal index')
    plt.ylabel('Balanced accuracy')
    plt.title('Hill-climb proposal trajectory')
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(12, 4))
    plt.step(
        log_df['proposal_index'],
        log_df['accepted_count_running'],
        where='post',
        color='black',
    )
    plt.xlabel('Proposal index')
    plt.ylabel('Accepted model count')
    plt.title('Accepted ensemble size over time')
    plt.tight_layout()
    plt.show()

if not trials_completed.empty:
    plt.figure(figsize=(12, 4))
    sns.histplot(trials_completed['value'], bins=20, color='lightgray', edgecolor='black')
    plt.xlabel('Trial objective value (median BA)')
    plt.ylabel('Trial count')
    plt.title('Optuna completed trial distribution')
    plt.tight_layout()
    plt.show()
else:
    print('No completed Optuna trials yet.')

if not accepted_specs_df.empty:
    plt.figure(figsize=(10, 4))
    sns.boxplot(data=accepted_specs_df[['row_fraction', 'feature_fraction', 'weight']], color='lightgray')
    plt.title('Accepted member sampling and weight distributions')
    plt.ylabel('Value')
    plt.tight_layout()
    plt.show()

if not accepted_params_df.empty:
    numeric_params = [
        column
        for column in accepted_params_df.columns
        if pd.api.types.is_numeric_dtype(accepted_params_df[column])
    ]
    if numeric_params:
        top_params = numeric_params[:6]
        plot_df = accepted_params_df[top_params].melt(var_name='parameter', value_name='value')
        plt.figure(figsize=(12, 4))
        sns.boxplot(data=plot_df, x='parameter', y='value', color='lightgray')
        plt.title('Accepted model parameter distributions')
        plt.xticks(rotation=25, ha='right')
        plt.tight_layout()
        plt.show()